## Open camera with OpenCV




In [6]:
import cv2 as c
cap = c.VideoCapture(0)
while True:
   ret, frame = cap.read()
   if not ret:
       break

   c.imshow("Camera", frame)

   if c.waitKey(1) == ord('q'):
       break

cap.release()
c.destroyAllWindows()



## Add MediaPipe (face points)

In [8]:
import cv2
import mediapipe as mp

mp_face = mp.solutions.face_mesh
face_mesh = mp_face.FaceMesh()

cap = cv2.VideoCapture(0)

while True:
   ret, frame = cap.read()
   if not ret:
       break

   rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
   result = face_mesh.process(rgb)

   if result.multi_face_landmarks:
       for face in result.multi_face_landmarks:
           for point in face.landmark:
               h, w, _ = frame.shape
               x = int(point.x * w)
               y = int(point.y * h)

               cv2.circle(frame, (x, y), 1, (0,255,0), -1)

   cv2.imshow("Face Points", frame)

   if cv2.waitKey(1) == ord('q'):
       break

cap.release()
cv2.destroyAllWindows()



## ONLY EYE POINTS

In [9]:
import cv2
import mediapipe as mp

mp_face = mp.solutions.face_mesh
face_mesh = mp_face.FaceMesh()

cap = cv2.VideoCapture(0)

OEIL_GAUCHE = [362, 385, 387, 263, 373, 380]

while True:
    ret, frame = cap.read()
    if not ret:
        break

    h, w, _ = frame.shape

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = face_mesh.process(rgb)

    if result.multi_face_landmarks:
        for face in result.multi_face_landmarks:
            for i in OEIL_GAUCHE:
                point = face.landmark[i]
                x = int(point.x * w)
                y = int(point.y * h)
                cv2.circle(frame, (x, y), 3, (0, 0, 255), -1)

    cv2.imshow("Eye Points Only", frame)

    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()




## BOTH EYES

In [10]:
import cv2
import mediapipe as mp

mp_face = mp.solutions.face_mesh
face_mesh = mp_face.FaceMesh()

cap = cv2.VideoCapture(0)

OEIL_GAUCHE = [362, 385, 387, 263, 373, 380]
OEIL_DROIT = [33, 160, 158, 133, 153, 144]

while True:
    ret, frame = cap.read()
    if not ret:
        break

    h, w, _ = frame.shape

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = face_mesh.process(rgb)

    if result.multi_face_landmarks:
        for face in result.multi_face_landmarks:

            # 👁️ LEFT EYE (RED)
            for i in OEIL_GAUCHE:
                point = face.landmark[i]
                x = int(point.x * w)
                y = int(point.y * h)
                cv2.circle(frame, (x, y), 3, (0, 0, 255), -1)

            # 👁️ RIGHT EYE (BLUE)
            for i in OEIL_DROIT:
                point = face.landmark[i]
                x = int(point.x * w)
                y = int(point.y * h)
                cv2.circle(frame, (x, y), 3, (255, 0, 0), -1)

    cv2.imshow("Both Eyes", frame)

    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

## DISTANCE FUNCTION TEST

In [12]:
import math

class Point:
   def __init__(self, x, y):
       self.x = x
       self.y = y
    

def distance(p1, p2):
   return math.sqrt((p1.x - p2.x)**2 + (p1.y - p2.y)**2)

# Test
p1 = Point(1, 2)
p2 = Point(4, 6)
print("Distance:", distance(p1, p2))

Distance: 5.0


## EAR DISPLAY

In [15]:
import cv2
import mediapipe as mp
import math

mp_face = mp.solutions.face_mesh
face_mesh = mp_face.FaceMesh()

cap = cv2.VideoCapture(0)

OEIL_GAUCHE = [362, 385, 387, 263, 373, 380]

def distance(p1, p2):
   return math.sqrt((p1.x - p2.x)**2 + (p1.y - p2.y)**2)

def calculer_ear(points, indices):
   p1, p2, p3, p4, p5, p6 = [points[i] for i in indices]
   return (distance(p2, p6) + distance(p3, p5)) / (2.0 * distance(p1, p4))

while True:
   ret, frame = cap.read()
   if not ret:
       break

   h, w, _ = frame.shape

   rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
   result = face_mesh.process(rgb)

   if result.multi_face_landmarks:
       for face in result.multi_face_landmarks:

           # draw eye
           for i in OEIL_GAUCHE:
               point = face.landmark[i]
               x = int(point.x * w)
               y = int(point.y * h)
               cv2.circle(frame, (x, y), 3, (0, 0, 255), -1)

           # EAR
           ear = calculer_ear(face.landmark, OEIL_GAUCHE)

           cv2.putText(frame, f"Distance: {ear:.2f}", (30, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

   cv2.imshow("EAR", frame)

   if cv2.waitKey(1) == ord('q'):
       break

cap.release()
cv2.destroyAllWindows()


## FATIGUE TIMER

In [18]:
import cv2
import mediapipe as mp
import math
import time

mp_face = mp.solutions.face_mesh
face_mesh = mp_face.FaceMesh()

cap = cv2.VideoCapture(0)

OEIL_GAUCHE = [362, 385, 387, 263, 373, 380]

def distance(p1, p2):
   return math.sqrt((p1.x - p2.x)**2 + (p1.y - p2.y)**2)

def calculer_ear(points, indices):
   p1, p2, p3, p4, p5, p6 = [points[i] for i in indices]
   return (distance(p2, p6) + distance(p3, p5)) / (2.0 * distance(p1, p4))

start_time = None

while True:
   ret, frame = cap.read()
   if not ret:
       break

   h, w, _ = frame.shape

   rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
   result = face_mesh.process(rgb)

   if result.multi_face_landmarks:
       for face in result.multi_face_landmarks:

           ear = calculer_ear(face.landmark, OEIL_GAUCHE)

           cv2.putText(frame, f"EAR: {ear:.2f}", (30, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

           if ear < 0.25:
               if start_time is None:
                   start_time = time.time()

               duration = time.time() - start_time

               cv2.putText(frame, f"Closed: {duration:.1f}s", (30, 60),
                           cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

               if duration >= 2:
                   cv2.putText(frame, "SLEEP!", (30, 100),
                               cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 3)
           else:
               start_time = None

   cv2.imshow("Fatigue Detection", frame)

   if cv2.waitKey(1) == ord('q'):
       break

cap.release()
cv2.destroyAllWindows()


## WITH SOUND

here is a simple pseudo code ( steps how the code gonna be )

#### libraries
Initialize MediaPipe (face points)
Initialize math (distance calculation)
Initialize sound (pygame)
#### start video
Start camera
#### start loop in the video
Loop:
    Capture frame
    Detect face points
    Get eye points
#### the logic behind is
    Calculate eye points distance using math

    If Eye_Dist < threshold:
        Start timer
        If time ≥ 2s:
            Play alarm
    Else:
        Stop alarm

    If 'q' pressed:
        Break
#### now the real code


In [19]:
import cv2
import mediapipe as mp
import math
import time
import pygame

pygame.mixer.init()
pygame.mixer.music.load("alarme.mp3")

mp_face = mp.solutions.face_mesh
face_mesh = mp_face.FaceMesh()

cap = cv2.VideoCapture(0)

OEIL_GAUCHE = [362, 385, 387, 263, 373, 380]

def distance(p1, p2):
   return math.sqrt((p1.x - p2.x)**2 + (p1.y - p2.y)**2)

def calculer_ear(points, indices):
   p1, p2, p3, p4, p5, p6 = [points[i] for i in indices]
   return (distance(p2, p6) + distance(p3, p5)) / (2.0 * distance(p1, p4))

start_time = None
alarm_on = False

while True:
   ret, frame = cap.read()
   if not ret:
       break

   rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
   result = face_mesh.process(rgb)

   if result.multi_face_landmarks:
       for face in result.multi_face_landmarks:

           ear = calculer_ear(face.landmark, OEIL_GAUCHE)

           if ear < 0.25:
               if start_time is None:
                   start_time = time.time()

               duration = time.time() - start_time

               if duration >= 2:
                   cv2.putText(frame, "Danger !!!", (30, 100),
                               cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 3)

                   if not alarm_on:
                       pygame.mixer.music.play(-1)
                       alarm_on = True
           else:
               start_time = None

               if alarm_on:
                   pygame.mixer.music.stop()
                   alarm_on = False

   cv2.imshow("Alarm System", frame)

   if cv2.waitKey(1) == ord('q'):
       break

cap.release()
cv2.destroyAllWindows()
pygame.mixer.quit()


pygame 2.6.1 (SDL 2.28.4, Python 3.11.0)
Hello from the pygame community. https://www.pygame.org/contribute.html


## with Arduino and buser

This version is to test the code with the arduino uno

-**what we need to install**

`opencv-python mediapipe pyserial`

-**and also write a simple code in arduino UNO to test the buzzer** and making sure to modify the com and the baud rate that arduino use


In [ ]:
import cv2
import mediapipe as mp
import math
import time
import serial

# ---------- Arduino ----------
arduino = serial.Serial('COM3', 9600)
time.sleep(2)

def buzzer_on():
    arduino.write(b'1')

def buzzer_off():
    arduino.write(b'0')

# ---------- MediaPipe ----------
mp_face = mp.solutions.face_mesh
detecteur = mp_face.FaceMesh()

# ---------- Camera ----------
camera = cv2.VideoCapture(0)

# ---------- Parameters ----------
SEUIL_EAR = 0.25
SEUIL_TEMPS_YEUX = 2.0

OEIL_GAUCHE = [362, 385, 387, 263, 373, 380]
OEIL_DROIT  = [33, 160, 158, 133, 153, 144]

# ---------- Utils ----------
def distance(p1, p2):
    return math.sqrt((p1.x - p2.x)**2 + (p1.y - p2.y)**2)

def calculer_ear(points, indices):
    p1, p2, p3, p4, p5, p6 = [points[i] for i in indices]
    return (distance(p2, p6) + distance(p3, p5)) / (2.0 * distance(p1, p4))

# ---------- Variables ----------
temps_yeux = None
alarme_active = False

# ---------- Main Loop ----------
while True:
    ret, image = camera.read()
    if not ret:
        break

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    resultats = detecteur.process(image_rgb)

    if resultats.multi_face_landmarks:
        for visage in resultats.multi_face_landmarks:
            points = visage.landmark

            ear_g = calculer_ear(points, OEIL_GAUCHE)
            ear_d = calculer_ear(points, OEIL_DROIT)
            ear = (ear_g + ear_d) / 2.0

            if ear < SEUIL_EAR:
                if temps_yeux is None:
                    temps_yeux = time.time()

                duree = time.time() - temps_yeux

                if duree >= SEUIL_TEMPS_YEUX:
                    cv2.putText(image, "SLEEP DETECTED!", (50, 100),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 3)

                    if not alarme_active:
                        buzzer_on()
                        alarme_active = True
            else:
                temps_yeux = None

                if alarme_active:
                    buzzer_off()
                    alarme_active = False

    cv2.imshow("DriveSafe", image)

    if cv2.waitKey(1) == ord('q'):
        break

# ---------- Cleanup ----------
camera.release()
cv2.destroyAllWindows()
buzzer_off()
arduino.close()

print("Done")